In [11]:

# graph (Dictionary of Dictionaries)

graph = {
    'S': {'A': 3, 'B': 6, 'C': 5},
    'A': {'D': 9, 'E': 8},
    'B': {'F': 12, 'G': 14},
    'C': {'H': 7},
    'H': {'I': 5, 'J': 6},
    'I': {'K': 1, 'L': 10, 'M': 2},
    'D': {}, 'E': {}, 'F': {}, 'G': {},
    'J': {}, 'K': {}, 'L': {}, 'M': {}
}

heuristic = {
    'S': 10, 'A': 9, 'B': 7, 'C': 5, 'D': 8, 'E': 6, 'F': 4, 'G': 3,
    'H': 3, 'I': 2, 'J': 6, 'K': 2, 'L': 0, 'M': 1
}

# start and goal (Strings)

start_node = 'S'
goal_node = 'G'

# heuristic (Dictionary)

# heuristic = {
#     'Start': 8,
#     'A': 5,
#     'B': 6,
#     'C': 2,
#     'D': 7,
#     'Goal': 0
# }

# frontier (List of Tuples)
# What it does: The "waiting list" of nodes to explore next.
# Each item is a tuple containing (node_name, priority_score).
# *Note: In real-world apps, use Python's `heapq` module instead of a list.

# frontier = [('C', 4), ('B', 7), ('D', 15)]

# visited (Set)
# What it does: Keeps track of nodes that have already been completely explored.
# Why a Set?: Optimized for "membership testing" (O(1) lookup time)
# to prevent infinite loops if your graph has cycles.

# visited = {'Start', 'A'}

# cost_so_far (Dictionary)
# What it does: Used in UCS and A*. Represents g(n).
cost_so_far = {
    # 'Start': 0,
    # 'A': 5,
    # 'B': 2
}


# ---------------------------------------------------------------------
# 3. THE BREADCRUMB TRAIL (Reconstructing the Answer)
# ---------------------------------------------------------------------

# came_from (Dictionary)
# What it does: Maps a node to its "parent" (the node that immediately
# preceded it on the best path). When the algorithm hits the goal,
# it loops backward through this dictionary until it hits the start node.
came_from = {
    # 'Start': None,
    # 'A': 'Start',
    # 'B': 'Start',
    # 'C': 'A',
    # 'Goal': 'C'
}

# path (List)
# final_path = ['Start', 'A', 'C', 'Goal']

# helper functions for A* ucs bfs

In [12]:
import heapq

# HELPER 1: Path Builder (Used by all 3)
def tracePath(curr, cameFrom):
    path = []
    while curr: path.append(curr); curr = cameFrom[curr]
    return path[::-1]

# HELPER 2: Neighbor Evaluator for UCS & A*
def getCheaper(curr, graph, costs, cameFrom):
    valid = []
    for nxt, edgeCost in graph.get(curr, {}).items():
        newCost = costs[curr] + edgeCost
        if newCost < costs.get(nxt, float('inf')):
            costs[nxt] = newCost
            cameFrom[nxt] = curr
            valid.append((newCost, nxt))
    return valid

# HELPER 3: Neighbor Evaluator for Greedy BFS
def getUnvisited(curr, graph, cameFrom):
    valid = []
    for nxt in graph.get(curr, {}):
        if nxt not in cameFrom:
            cameFrom[nxt] = curr
            valid.append(nxt)
    return valid

## BFS

In [13]:
def greedyBfs(graph, start, goal, h):
    frontier = [(h[start], start)]
    cameFrom = {start: None}

    while frontier:
        _, curr = heapq.heappop(frontier)
        if curr == goal: return tracePath(curr, cameFrom)
        for nxt in getUnvisited(curr, graph, cameFrom):
            # Priority = h(n)
            heapq.heappush(frontier, (h[nxt], nxt))
    return None

## hill climbing

In [14]:
def hillClimbing(graph, start, goal, h):
    curr = start
    cameFrom = {start: None}

    while curr:
        if curr == goal:
            return tracePath(curr, cameFrom)

        # 1. Look at all immediate neighbors
        neighbors = list(graph.get(curr, {}).keys())

        if not neighbors:
            print("Hit a dead end. No path found.")
            return None

        # 2. Pick the absolute best neighbor based purely on heuristic
        bestNxt = min(neighbors, key=lambda n: h[n])

        # 3. The Local Maximum Check (The core Hill Climbing logic)
        if h[bestNxt] >= h[curr]:
            print("Stuck at a local maximum! Cannot move closer to goal.")
            return None

        # 4. Move forward
        # (Notice we don't save the other neighbors to a frontier. They are deleted forever!)
        cameFrom[bestNxt] = curr
        curr = bestNxt

    return None

## UCS

In [15]:
def ucs(graph, start, goal):
    frontier = [(0, start)]
    cameFrom = {start: None}
    costs = {start: 0}

    while frontier:
        _, curr = heapq.heappop(frontier)

        if curr == goal: return tracePath(curr, cameFrom)

        for newCost, nxt in getCheaper(curr, graph, costs, cameFrom):
            # Priority = g(n)
            heapq.heappush(frontier, (newCost, nxt))

    return None

## Beam search

In [16]:
def beamSearchUcs(graph, start, goal, k):
    frontier = [(0, start)] # Priority is g(n)
    cameFrom = {start: None}
    costs = {start: 0}

    while frontier:
        nextFrontier = []

        # 1. Expand ALL nodes currently in the beam
        for currentCost, curr in frontier:

            if curr == goal:
                return tracePath(curr, cameFrom)

            # 2. Use our helper to gather all valid neighbors
            neighbors = getCheaper(curr, graph, costs, cameFrom)
            nextFrontier.extend(neighbors)

        # 3. The Beam Cut: Sort the massive list of new neighbors and keep only top 'k'
        nextFrontier.sort(key=lambda x: x[0])
        frontier = nextFrontier[:k]

    return None

## A*

In [17]:
def aStar(graph, start, goal, h):
    frontier = [(h[start], start)]
    cameFrom = {start: None}
    costs = {start: 0}

    while frontier:
        _, curr = heapq.heappop(frontier)

        if curr == goal: return tracePath(curr, cameFrom)

        for newCost, nxt in getCheaper(curr, graph, costs, cameFrom):
            # Priority = g(n) + h(n)
            heapq.heappush(frontier, (newCost + h[nxt], nxt))

    return None

In [18]:
graph = {
    'S': {'A': 1, 'B': 4},
    'A': {'B': 2, 'C': 5, 'L': 12},
    'B': {'C': 2},
    'C': {'L': 3},
    'L': {}
}

heuristic = {
    'S': 7,
    'A': 6,
    'B': 2,
    'C': 1,
    'L': 0
}

startNode = 'S'
goalNode = 'L'
k = 2  # Beam width for Beam Search
print("Greedy BFS:   ", greedyBfs(graph, startNode, goalNode, heuristic))
print("Hill Climbing:", hillClimbing(graph, startNode, goalNode, heuristic))
print("UCS:          ", ucs(graph, startNode, goalNode))
print("Beam Search:  ", beamSearchUcs(graph, startNode, goalNode, k))
print("A* Search:    ", aStar(graph, startNode, goalNode, heuristic))

Greedy BFS:    ['S', 'B', 'C', 'L']
Hill Climbing: ['S', 'B', 'C', 'L']
UCS:           ['S', 'A', 'B', 'C', 'L']
Beam Search:   ['S', 'A', 'B', 'C', 'L']
A* Search:     ['S', 'A', 'B', 'C', 'L']


## combination of UCS BFS A*

In [19]:
import heapq

def masterSearch(graph, start, goal, h):
    # ==========================================
    # 1. INITIALIZE THE FRONTIER & TRACKERS
    # ==========================================
    # frontier = [(h[start], start)]   # BFS A*
    # frontier = [(0, start)]          # UCS
    cameFrom = {start: None}           # ALL
    # costs = {start: 0}               # UCS A*

    # ==========================================
    # 2. THE CORE ENGINE (Runs for all 3)
    # ==========================================
    while frontier:                    # ALL
        _, curr = heapq.heappop(frontier) # ALL

        if curr == goal:               # ALL
            return tracePath(curr, cameFrom) # ALL (HELPER 1)

        # ==========================================
        # 3. EVALUATE NEIGHBORS
        # ==========================================

        # --- GREEDY BFS LOGIC --- #
        # for nxt in getUnvisited(curr, graph, cameFrom):        # BFS (HELPER 3)
            # heapq.heappush(frontier, (h[nxt], nxt))            # BFS

        # --- UCS & A* LOGIC --- #
        # for newCost, nxt in getCheaper(curr, graph, costs, cameFrom): # UCS A* (HELPER 2)
            # heapq.heappush(frontier, (newCost, nxt))           # UCS
            # heapq.heappush(frontier, (newCost + h[nxt], nxt))  # A*

    return None                        # ALL

## GA

In [20]:
import random

# ==========================================
# 1. THE GENETIC OPERATORS (Helper Functions)
# ==========================================

def evaluate(chrom):
    """Calculates fitness (Goal: maximize number of 1s)"""
    return sum(chrom)

def selectParents(pop, fitnesses):
    """Roulette Wheel Selection: higher fitness = higher chance to be picked"""
    return random.choices(pop, weights=fitnesses, k=2)

def crossover(p1, p2):
    """Single-point crossover: slices and swaps parent genes"""
    pt = random.randint(1, len(p1) - 1)
    return p1[:pt] + p2[pt:], p2[:pt] + p1[pt:]

def mutate(chrom, rate):
    """Bit-flip mutation: randomly flips 0s to 1s and vice versa"""
    return [(1 - bit if random.random() < rate else bit) for bit in chrom]


# ==========================================
# 2. THE CORE ENGINE
# ==========================================

def geneticAlgo(pop, gens, mutRate):

    for _ in range(gens):
        nextGen = []

        # --- 1. EVALUATE ---
        fitnesses = [evaluate(chrom) for chrom in pop]

        # --- 2. BREED ---
        for _ in range(len(pop) // 2):

            # A. SELECT parents
            p1, p2 = selectParents(pop, fitnesses)

            # B. CROSSOVER
            c1, c2 = crossover(p1, p2)

            # C. MUTATE & ADD
            nextGen.extend([mutate(c1, mutRate), mutate(c2, mutRate)])

        # --- 3. REPLACE ---
        pop = nextGen

    return max(pop, key=evaluate)


# ==========================================
# 3. EXECUTION
# ==========================================

initPop = [
    [0, 1, 1, 0, 1],
    [1, 1, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 1, 1]
]

bestSol = geneticAlgo(initPop, gens=50, mutRate=0.01)

print("Best solution:", bestSol)
print("Fitness:", evaluate(bestSol))

Best solution: [0, 1, 1, 1, 1]
Fitness: 4
